# Process Path Exploration and QA
A process path is defined here as a list of processes from the current to the root.

## Setup
Download data from https://gdo-wintap.llnl.gov

A good starting set is: https://gdo168.llnl.gov/data/ACME-2023/stdview-20231105-20231120/process_path.parquet

If you'd like more data, look into the longer date ranges.

Modify the path in the "create view" statement to point to where you have downloaded the process_summary.parquet file. 


In [2]:
# Import packages used in notebooks
import altair as alt
import duckdb
import pandas as pd
%load_ext magic_duckdb

pd.set_option('display.max_rows', 200)


In [3]:
# Initialize an in-memory db. Save reference in a variable and then set magic-duckdb environment. Result is ability to use the same DB instance from python code and %dql/%%dql magics.
con = duckdb.connect()
%dql -co con
# Only uses a process_path table
%dql create view process_path as from '~/data/wintapv6/ACME-Redo/stdview-20231105-20231120/process_path.parquet'
# Display a simple summary of the table
%dql summarize process_path


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,agent_id,VARCHAR,None,None,0,None,None,None,None,None,4707705,100.00
1,hostname,VARCHAR,ACME-DC1,ACME-WS-UVF,23,None,None,None,None,None,4707705,0.00
2,pid_hash,VARCHAR,00000016049E7D08D555AA56CD072E2F,FFFFFB83788869D71024896A7A0AB5E3,5127288,None,None,None,None,None,4707705,0.00
3,os_pid,INTEGER,0,17404,3997,5881.477275020418,3749.8624803398047,3141,5072,7983,4707705,0.00
4,process_name,VARCHAR,/powershell.exe,xmlcatalog.exe,640,None,None,None,None,None,4707705,1.88
5,process_path,VARCHAR,%systemroot%\system32\csrss.exe,wlrmdr.exe,2030,None,None,None,None,None,4707705,1.88
6,level,INTEGER,0,448,482,6.242456356122569,4.700774239759703,6,6,7,4707705,0.00
7,parent_pid_hash,VARCHAR,0004A3028DF9B41A327607921D373D28,FFB335D887A9496741C2E609899391BB,391,None,None,None,None,None,4707705,1.89
8,parent_os_pid,INTEGER,4,11728,284,283.65467104382833,548.9178043348807,4,4,614,4707705,1.89
9,seq,INTEGER,1,449,499,7.242456356122569,4.700774239759703,7,7,8,4707705,0.00


# Process Paths: Initial EDA
Build a base SQL view that parses the path structure. This just simplifies the queries after this point.

Features:
* root_process_name
* root_pid_hash
* parent_process_name
* grand_parent_process_name
* great_grand_parent_process_name




In [4]:
%%dql
create or replace view process_path_root
as
select
	*,
	ptree_list_tuples[-1:][1]['process_name'] root_process_name,
	ptree_list_tuples[-1:][1]['pid_hash'] root_pid_hash,
	ptree_list_tuples[2:2][1]['process_name'] parent_process_name,
	ptree_list_tuples[2:2][1]['pid_hash'] parent_pid_hash,
	ptree_list_tuples[3:3][1]['process_name'] grand_parent_process_name,
	ptree_list_tuples[3:3][1]['pid_hash'] grand_parent_pid_hash,
	ptree_list_tuples[4:4][1]['process_name'] great_grand_parent_process_name,
	ptree_list_tuples[4:4][1]['pid_hash'] great_grand_parent_pid_hash
from
	process_path


,Count


In [5]:
%%dql
-- What are the roots? They *should* all be ntoskrnl.exe
select
	root_process_name,
	count(distinct hostname) num_hosts,
	count(distinct root_pid_hash) uniq_root_pid_hash,
	median(level) median_path_depth,
	max(level) max_path_depth,
	count(*) num_process
from
	process_path_root
group by all
order by num_hosts


,root_process_name,num_hosts,uniq_root_pid_hash,median_path_depth,max_path_depth,num_process
0,cmd.exe,1,155,155.0,448,1987
1,devenv.exe,1,1,8.0,20,1243
2,searchapp.exe,1,1,4.0,5,875
3,vshost.exe,1,1,7.0,9,411
4,shellexperiencehost.exe,1,1,5.0,10,666
5,taskkill.exe,1,1,0.0,0,1
6,msedge.exe,1,2,5.0,6,222
7,vcpkgsrv.exe,1,1,6.0,6,45
8,wmiprvse.exe,2,2,5.0,7,119
9,logonui.exe,2,2,5.0,6,70


In [6]:
%%dql
-- What happens if we remove mergehelper and wintap?
select
	root_process_name,
	count(distinct hostname) num_hosts,
	count(distinct root_pid_hash) uniq_root_pid_hash,
	median(level) median_path_depth,
	max(level) max_path_depth,
	count(*) num_process
from
	process_path_root
    where process_name not in ('mergehelper.exe','wintap.exe')
group by all
order by num_hosts


,root_process_name,num_hosts,uniq_root_pid_hash,median_path_depth,max_path_depth,num_process
0,cmd.exe,1,155,155.0,448,1987
1,vshost.exe,1,1,7.0,9,411
2,searchapp.exe,1,1,4.0,5,875
3,devenv.exe,1,1,8.0,20,1243
4,shellexperiencehost.exe,1,1,5.0,10,666
5,msedge.exe,1,2,5.0,6,222
6,vcpkgsrv.exe,1,1,6.0,6,45
7,taskkill.exe,1,1,0.0,0,1
8,userinit.exe,2,2,5.5,6,22
9,logonui.exe,2,2,5.0,6,70


In [7]:
%%dql
-- What are the roots? They *should* all be ntoskrnl.exe
select *, 
   	-- Calculate percent for host and rank within host
	RANK() OVER (PARTITION BY hostname ORDER BY num_process DESC) rank_pos,
    round((num_process/sum(num_process) OVER (PARTITION BY hostname))*100,2) root_process_name_pct
from (
	select
		hostname, 
		root_process_name,
		count(distinct root_pid_hash) uniq_root_pid_hash,
		median(level) median_path_depth,
		max(level) max_path_depth,
		count(*) num_process,
	from
		process_path_root
	group by all)
order by hostname, num_process


,hostname,root_process_name,uniq_root_pid_hash,median_path_depth,max_path_depth,num_process,rank_pos,root_process_name_pct
0,ACME-DC1,wininit.exe,1,2.5,3,2,6,0.00
1,ACME-DC1,services.exe,1,3.0,3,3,5,0.00
2,ACME-DC1,ntoskrnl.exe,1,10.0,127,1529,4,0.83
3,ACME-DC1,cmd.exe,155,155.0,448,1987,3,1.08
4,ACME-DC1,None,4429,0.0,1,4433,2,2.42
5,ACME-DC1,svchost.exe,1,5.0,67,175410,1,95.66
6,ACME-HH-AKA,conhost.exe,1,0.0,0,1,5,0.00
7,ACME-HH-AKA,winlogon.exe,1,1.5,2,2,4,0.00
8,ACME-HH-AKA,logonui.exe,1,2.0,2,3,3,0.00
9,ACME-HH-AKA,None,2657,0.0,1,2665,2,1.99
